In [1]:
import PyPDF2

with open("/Users/animeshsingh/Desktop/Datasets/Mental Health/anxity.pdf", "rb") as f:
    reader = PyPDF2.PdfReader(f)
    anxiety_text = ""
    for page in reader.pages:
        anxiety_text += page.extract_text()

print(len(anxiety_text))

2599850


In [2]:
with open("/Users/animeshsingh/Desktop/Datasets/Mental Health/ocd.pdf", "rb") as f:
    reader = PyPDF2.PdfReader(f)
    ocd_text = ""
    for page in reader.pages:
        ocd_text += page.extract_text()

print(len(ocd_text))

4254895


In [3]:
# maybe try ocr
with open("/Users/animeshsingh/Desktop/Datasets/Mental Health/depresion.pdf", "rb") as f:
    reader = PyPDF2.PdfReader(f)
    depression_text = ""
    for page in reader.pages:
        depression_text += page.extract_text()

print(len(depression_text))

3293463


In [4]:
with open("/Users/animeshsingh/Desktop/Datasets/Mental Health/bipolar.pdf", "rb") as f:
    reader = PyPDF2.PdfReader(f)
    bipolar_text = ""
    for page in reader.pages:
        bipolar_text += page.extract_text()

print(len(bipolar_text))

6643859


In [5]:
with open("/Users/animeshsingh/Desktop/Datasets/Mental Health/schiz.pdf", "rb") as f:
    reader = PyPDF2.PdfReader(f)
    schiz_text = ""
    for page in reader.pages:
        schiz_text += page.extract_text()

print(len(schiz_text))

1439832


In [6]:
def lowerc(x):
    return x.lower()

anxiety_text = lowerc(anxiety_text)
bipolar_text = lowerc(bipolar_text)
ocd_text = lowerc(ocd_text)
depression_text = lowerc(depression_text)
schiz_text = lowerc(schiz_text)

In [7]:
import re

def punc(x):
    return re.sub(r"[^A-Za-z\s:.]", "", x)

anxiety_text = punc(anxiety_text)
bipolar_text = punc(bipolar_text)
ocd_text = punc(ocd_text)
depression_text = punc(depression_text)
schiz_text = punc(schiz_text)

In [8]:
def remove_nl(x):
    return re.sub(r"\n+", " ", x)

anxiety_text = remove_nl(anxiety_text)
bipolar_text = remove_nl(bipolar_text)
ocd_text = remove_nl(ocd_text)
depression_text = remove_nl(depression_text)
schiz_text = remove_nl(schiz_text)

In [ ]:
# depression_text = depression_text[1000:10000]
# anxiety_text = anxiety_text[1000:10000]
# bipolar_text = bipolar_text[1000:10000]
# ocd_text = ocd_text[1000:10000]
# schiz_text = schiz_text[1000:10000]

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from datasets import Dataset

save_path = "/Users/animeshsingh/Desktop/Models/Llama 3.2 1b"

tokenizer = AutoTokenizer.from_pretrained("/Users/animeshsingh/Desktop/Models/Llama 3.2 1b")
model = AutoModelForCausalLM.from_pretrained("/Users/animeshsingh/Desktop/Models/Llama 3.2 1b")
device = "mps"
model.to(device)

tokenizer.pad_token = tokenizer.eos_token

The module name Llama 3_dot_2 1b (originally Llama 3.2 1b) is not a valid Python identifier. Please rename the original module to avoid import issues.


In [11]:
depression_dict = Dataset.from_dict({"text": depression_text.split(". ")})
anxiety_dict = Dataset.from_dict({"text": anxiety_text.split(". ")})
bipolar_dict = Dataset.from_dict({"text": bipolar_text.split(". ")})
ocd_dict = Dataset.from_dict({"text": ocd_text.split(". ")})
schiz_dict = Dataset.from_dict({"text": schiz_text.split(". ")})

In [12]:
bs = 64

def tokenize(dicts):
    return tokenizer(dicts["text"], truncation = True, padding = "max_length", max_length = bs)

tokenized_depression = depression_dict.map(tokenize, batched = True, remove_columns = ["text"])
tokenized_anxiety = anxiety_dict.map(tokenize, batched = True, remove_columns = ["text"])
tokenized_bipolar = bipolar_dict.map(tokenize, batched = True, remove_columns = ["text"])
tokenized_ocd = ocd_dict.map(tokenize, batched = True, remove_columns = ["text"])
tokenized_schiz = schiz_dict.map(tokenize, batched = True, remove_columns = ["text"])

def grp(examples):
    chunk = []
    for i in examples["input_ids"]:
        chunk.extend(i)
    
    chunk_length = (len(chunk) // bs) * bs
    chunk = chunk[:chunk_length]
    grpd = {
        "input_ids": [chunk[i:i + bs] for i in range(0, chunk_length, bs)]
    }
    grpd['labels'] = grpd['input_ids'].copy()
    return grpd

depression_grpd = tokenized_depression.map(grp, batched = True)
anxiety_grpd = tokenized_anxiety.map(grp, batched = True)
bipolar_grpd = tokenized_bipolar.map(grp, batched = True)
ocd_grpd = tokenized_ocd.map(grp, batched = True)   
schiz_grpd = tokenized_schiz.map(grp, batched = True)

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

Map:   0%|          | 0/62 [00:00<?, ? examples/s]

Map:   0%|          | 0/97 [00:00<?, ? examples/s]

Map:   0%|          | 0/102 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

Map:   0%|          | 0/62 [00:00<?, ? examples/s]

Map:   0%|          | 0/97 [00:00<?, ? examples/s]

Map:   0%|          | 0/102 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [13]:
depression_grpd

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 154
})

In [14]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 425,984 || all params: 1,236,240,384 || trainable%: 0.0345


In [ ]:
from transformers import TrainingArguments, Trainer

train_args = TrainingArguments(
    output_dir = "/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters",
    gradient_accumulation_steps = 8,
    num_train_epochs = 2,
    logging_steps = 4,
    learning_rate = 1e-3,
    per_device_train_batch_size = 4,
    fp16 = False,
    optim = "adamw_torch",
    report_to = "none"
)

In [16]:
trainer = Trainer(
    model=model,
    args = train_args,
    train_dataset=depression_grpd,
)
trainer.train()

model.save_pretrained("/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/depression")
tokenizer.save_pretrained("/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/depression")

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
4,5.302700
8,1.437500


('/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/depression/tokenizer_config.json',
 '/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/depression/special_tokens_map.json',
 '/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/depression/tokenizer.json')

In [17]:
trainer = Trainer(
    model=model,
    args = train_args,
    train_dataset=anxiety_grpd,
)
trainer.train()

model.save_pretrained("/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/anxiety")
tokenizer.save_pretrained("/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/anxiety")

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
4,1.765200


('/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/anxiety/tokenizer_config.json',
 '/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/anxiety/special_tokens_map.json',
 '/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/anxiety/tokenizer.json')

In [18]:
trainer = Trainer(
    model=model,
    args = train_args,
    train_dataset=bipolar_grpd,
)
trainer.train()

model.save_pretrained("/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/bipolar")
tokenizer.save_pretrained("/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/bipolar")

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
4,1.241600
8,1.119500


('/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/bipolar/tokenizer_config.json',
 '/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/bipolar/special_tokens_map.json',
 '/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/bipolar/tokenizer.json')

In [19]:
trainer = Trainer(
    model=model,
    args = train_args,
    train_dataset=ocd_grpd,
)
trainer.train()

model.save_pretrained("/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/ocd")
tokenizer.save_pretrained("/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/ocd")

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
4,1.396200
8,1.157000


('/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/ocd/tokenizer_config.json',
 '/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/ocd/special_tokens_map.json',
 '/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/ocd/tokenizer.json')

In [20]:
trainer = Trainer(
    model=model,
    args = train_args,
    train_dataset=schiz_grpd,
)
trainer.train()

model.save_pretrained("/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/schiz")
tokenizer.save_pretrained("/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/schiz")

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
4,1.181400
8,1.010000


('/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/schiz/tokenizer_config.json',
 '/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/schiz/special_tokens_map.json',
 '/Users/animeshsingh/Desktop/Projects/MoE_exp/adapters/schiz/tokenizer.json')